In [1]:
# Feature Engineering
# Feature engineering transforms raw data into signals that machines and humans can act on. In AIOps — raw cpu_pct is a number. cpu_above_baseline, is_spike, server_risk_score are actionable features.

In [2]:
# Load datasets
import pandas as pd
df_metrics = pd.read_csv("server_metrics.csv")
df_tickets = pd.read_csv("incidents.csv")
df_logs = pd.read_csv("app_logs.csv")

In [3]:
# Ratio features

In [4]:
df_metrics_fe = df_metrics.drop_duplicates(
    subset=["server_id", "timestamp"]
).sort_values(["server_id", "timestamp"]).copy()

# CPU to memory ratio — high CPU + low memory = different problem than both high
df_metrics_fe["cpu_memory_ratio"] = (
    df_metrics_fe["cpu_pct"] / df_metrics_fe["memory_pct"]
).round(3)

# Disk utilization ratio — how close to full
df_metrics_fe["disk_pressure"] = (
    df_metrics_fe["disk_pct"] / 100
).round(3)

print(df_metrics_fe[["server_id", "cpu_pct", "memory_pct", 
                       "cpu_memory_ratio", "disk_pressure"]].head(10))
# AIOps use — ratio features catch imbalanced resource usage that absolute values miss.

   server_id  cpu_pct  memory_pct  cpu_memory_ratio  disk_pressure
0     srv-01     49.6        94.6             0.524          0.689
5     srv-01     82.0        43.6             1.881          0.685
10    srv-01     96.6        82.7             1.168          0.882
15    srv-01     77.6        82.4             0.942          0.533
20    srv-01     22.5        73.3             0.307          0.631
25    srv-01     53.7        85.6             0.627          0.305
30    srv-01     91.8        46.3             1.983          0.618
35    srv-01     33.8        77.0             0.439          0.909
40    srv-01     70.7        35.7             1.980          0.884
45    srv-01     39.3        96.2             0.409          0.880


In [5]:
# Lag features
# AIOps use — lag features are the foundation of time series ML models. "CPU 1 hour ago" is often the strongest predictor of "CPU now".


In [6]:
# Previous reading value — what was CPU 1 hour ago?
df_metrics_fe["cpu_lag1"] = df_metrics_fe.groupby(
    "server_id", observed=True
)["cpu_pct"].shift(1)

# 3 hours ago
df_metrics_fe["cpu_lag3"] = df_metrics_fe.groupby(
    "server_id", observed=True
)["cpu_pct"].shift(3)

# Change from 1 hour ago
df_metrics_fe["cpu_change_1h"] = (
    df_metrics_fe["cpu_pct"] - df_metrics_fe["cpu_lag1"]
).round(2)

print(df_metrics_fe[["server_id", "timestamp", "cpu_pct", 
                       "cpu_lag1", "cpu_lag3", "cpu_change_1h"]].head(15))


   server_id            timestamp  cpu_pct  cpu_lag1  cpu_lag3  cpu_change_1h
0     srv-01  2026-01-01 00:00:00     49.6       NaN       NaN            NaN
5     srv-01  2026-01-01 01:00:00     82.0      49.6       NaN           32.4
10    srv-01  2026-01-01 02:00:00     96.6      82.0       NaN           14.6
15    srv-01  2026-01-01 03:00:00     77.6      96.6      49.6          -19.0
20    srv-01  2026-01-01 04:00:00     22.5      77.6      82.0          -55.1
25    srv-01  2026-01-01 05:00:00     53.7      22.5      96.6           31.2
30    srv-01  2026-01-01 06:00:00     91.8      53.7      77.6           38.1
35    srv-01  2026-01-01 07:00:00     33.8      91.8      22.5          -58.0
40    srv-01  2026-01-01 08:00:00     70.7      33.8      53.7           36.9
45    srv-01  2026-01-01 09:00:00     39.3      70.7      91.8          -31.4
50    srv-01  2026-01-01 10:00:00     43.3      39.3      33.8            4.0
55    srv-01  2026-01-01 11:00:00     59.6      43.3      70.7  

In [7]:
# Deviation from server baseline

# AIOps use — deviation from baseline is more meaningful than raw value. srv-04 at 80% CPU might be normal if its baseline is 75%. 
# srv-01 at 80% is critical if its baseline is 50%

In [8]:
# How far is each reading from this server's overall mean?
df_metrics_fe["cpu_server_mean"] = df_metrics_fe.groupby(
    "server_id", observed=True
)["cpu_pct"].transform("mean")

df_metrics_fe["cpu_deviation"] = (
    df_metrics_fe["cpu_pct"] - df_metrics_fe["cpu_server_mean"]
).round(2)

# Normalize deviation — % above or below baseline
df_metrics_fe["cpu_deviation_pct"] = (
    df_metrics_fe["cpu_deviation"] / df_metrics_fe["cpu_server_mean"] * 100
).round(2)

print(df_metrics_fe[["server_id", "cpu_pct", "cpu_server_mean", 
                       "cpu_deviation", "cpu_deviation_pct"]].head(10))

   server_id  cpu_pct  cpu_server_mean  cpu_deviation  cpu_deviation_pct
0     srv-01     49.6            59.64         -10.04             -16.83
5     srv-01     82.0            59.64          22.36              37.49
10    srv-01     96.6            59.64          36.96              61.97
15    srv-01     77.6            59.64          17.96              30.11
20    srv-01     22.5            59.64         -37.14             -62.27
25    srv-01     53.7            59.64          -5.94              -9.96
30    srv-01     91.8            59.64          32.16              53.92
35    srv-01     33.8            59.64         -25.84             -43.33
40    srv-01     70.7            59.64          11.06              18.54
45    srv-01     39.3            59.64         -20.34             -34.10


In [9]:
# Time based features

# AIOps use — incidents during business hours vs weekends have different severity and response patterns. 
# Time features let ML models learn these patterns.

In [10]:
# Extract time components — useful for pattern detection
df_metrics_fe["timestamp"] = pd.to_datetime(df_metrics_fe["timestamp"])
df_metrics_fe["hour"] = df_metrics_fe["timestamp"].dt.hour
df_metrics_fe["day_of_week"] = df_metrics_fe["timestamp"].dt.dayofweek
df_metrics_fe["is_business_hours"] = df_metrics_fe["hour"].between(9, 18).astype(int)
df_metrics_fe["is_weekend"] = (df_metrics_fe["day_of_week"] >= 5).astype(int)

print(df_metrics_fe[["server_id", "timestamp", "hour", 
                       "day_of_week", "is_business_hours", "is_weekend"]].head(10))

   server_id           timestamp  hour  day_of_week  is_business_hours  \
0     srv-01 2026-01-01 00:00:00     0            3                  0   
5     srv-01 2026-01-01 01:00:00     1            3                  0   
10    srv-01 2026-01-01 02:00:00     2            3                  0   
15    srv-01 2026-01-01 03:00:00     3            3                  0   
20    srv-01 2026-01-01 04:00:00     4            3                  0   
25    srv-01 2026-01-01 05:00:00     5            3                  0   
30    srv-01 2026-01-01 06:00:00     6            3                  0   
35    srv-01 2026-01-01 07:00:00     7            3                  0   
40    srv-01 2026-01-01 08:00:00     8            3                  0   
45    srv-01 2026-01-01 09:00:00     9            3                  1   

    is_weekend  
0            0  
5            0  
10           0  
15           0  
20           0  
25           0  
30           0  
35           0  
40           0  
45           0 

In [11]:
# Composite risk score
# single risk score per reading feeds dashboards, alerting engines, and ML classifiers. Normalization ensures no single metric dominates.

In [12]:
# Normalize each metric to 0-1 range
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df_metrics_fe["cpu_norm"]      = normalize(df_metrics_fe["cpu_pct"])
df_metrics_fe["memory_norm"]   = normalize(df_metrics_fe["memory_pct"])
df_metrics_fe["response_norm"] = normalize(df_metrics_fe["response_ms"])
df_metrics_fe["disk_norm"]     = normalize(df_metrics_fe["disk_pct"])

# Weighted composite risk score
df_metrics_fe["risk_score"] = (
    df_metrics_fe["cpu_norm"]      * 0.35 +
    df_metrics_fe["memory_norm"]   * 0.25 +
    df_metrics_fe["response_norm"] * 0.25 +
    df_metrics_fe["disk_norm"]     * 0.15
).round(3)

print(df_metrics_fe.groupby("server_id", observed=True)["risk_score"].mean().round(3).sort_values(ascending=False))

server_id
srv-05    0.505
srv-01    0.503
srv-02    0.502
srv-03    0.496
srv-04    0.490
Name: risk_score, dtype: float64


In [13]:
# Key rules
# Rule                                         Why
# --------                                      ------
# Always .copy() before feature engineering    Never mutate original DataFrame
# Normalize before composite scoring           Prevents one metric dominating
# Use astype(int) for binary flags             Cleaner than True/False for scoring
# Per-server deviation over absolute value     Context-aware features always win
# Lag features require sort by timestamp       Otherwise lags are meaningless

In [14]:
# Binary flag features

In [15]:
# Convert continuous metrics to actionable binary flags
df_metrics_fe["cpu_critical"]    = (df_metrics_fe["cpu_pct"] > 85).astype(int)
df_metrics_fe["memory_critical"] = (df_metrics_fe["memory_pct"] > 85).astype(int)
df_metrics_fe["disk_warning"]    = (df_metrics_fe["disk_pct"] > 75).astype(int)
df_metrics_fe["response_slow"]   = (df_metrics_fe["response_ms"] > 900).astype(int)

# How many critical flags at once — multi-resource stress indicator
df_metrics_fe["stress_count"] = (
    df_metrics_fe["cpu_critical"] +
    df_metrics_fe["memory_critical"] +
    df_metrics_fe["disk_warning"] +
    df_metrics_fe["response_slow"]
)

print(df_metrics_fe["stress_count"].value_counts().sort_index())
print(df_metrics_fe[df_metrics_fe["stress_count"] >= 3][
    ["server_id", "timestamp", "cpu_pct", "memory_pct", 
     "disk_pct", "response_ms", "stress_count"]
].head(10))

stress_count
0    186
1    191
2    101
3     19
4      3
Name: count, dtype: int64
    server_id           timestamp  cpu_pct  memory_pct  disk_pct  response_ms  \
10     srv-01 2026-01-01 02:00:00     96.6        82.7      88.2       1130.4   
210    srv-01 2026-01-02 18:00:00     89.1        66.0      81.9       1129.9   
220    srv-01 2026-01-02 20:00:00     24.6        95.9      90.3       1066.4   
310    srv-01 2026-01-03 14:00:00     95.2        70.9      87.5        992.1   
370    srv-01 2026-01-04 02:00:00     42.6        85.5      85.0       1040.6   
101    srv-02 2026-01-01 20:00:00     32.8        91.9      91.7        995.9   
162    srv-03 2026-01-02 08:00:00     97.8        87.0      89.9        193.4   
212    srv-03 2026-01-02 18:00:00     89.0        96.9      57.2        933.5   
437    srv-03 2026-01-04 15:00:00     94.1        89.2      81.2        931.0   
48     srv-04 2026-01-01 09:00:00     94.3        94.9      54.1       1102.1   

     stress_count  
10  

In [16]:
# Task 1 — server_metrics
# Create lag features cpu_lag1 and cpu_lag3. Add cpu_change_1h column. Which server has the highest average 1-hour CPU change — most volatile server?

In [17]:
# Sort properly first
df_metrics_sorted = df_metrics.sort_values(
    ["server_id", "timestamp"]
)

# Create lag features
df_metrics_sorted["cpu_lag1"] = (
    df_metrics_sorted
    .groupby("server_id")["cpu_pct"]
    .shift(1)
)

df_metrics_sorted["cpu_lag3"] = (
    df_metrics_sorted
    .groupby("server_id")["cpu_pct"]
    .shift(3)
)

# 1-hour CPU change
# Assuming data collected every 10 minutes:
# 6 rows ≈ 1 hour

df_metrics_sorted["cpu_change_1h"] = (
    df_metrics_sorted["cpu_pct"]
    -
    df_metrics_sorted
    .groupby("server_id")["cpu_pct"]
    .shift(6)
)

# Average volatility per server
volatility = (
    df_metrics_sorted
    .groupby("server_id")["cpu_change_1h"]
    .mean()
    .abs()
    .sort_values(ascending=False)
)

print(volatility)

# Most volatile server
print("\nMost volatile server:")
print(volatility.idxmax())

server_id
srv-05    1.525253
srv-03    0.966667
srv-01    0.770707
srv-02    0.640404
srv-04    0.580808
Name: cpu_change_1h, dtype: float64

Most volatile server:
srv-05


In [18]:
# Task 2 — server_metrics
# Build a composite risk score using normalized cpu_pct, memory_pct, response_ms, disk_pct with weights 0.35, 0.25, 0.25, 0.15. Rank servers by mean risk score. Does ranking match Topic 6 results?

In [19]:
# Sort
df_metrics_sorted = df_metrics.sort_values(["server_id", "timestamp"])

# Lag features
df_metrics_sorted["cpu_lag1"] = (
    df_metrics_sorted.groupby("server_id")["cpu_pct"].shift(1)
)

df_metrics_sorted["cpu_lag3"] = (
    df_metrics_sorted.groupby("server_id")["cpu_pct"].shift(3)
)

# 1-hour change (assuming 10-min interval = 6 steps)
df_metrics_sorted["cpu_change_1h"] = (
    df_metrics_sorted["cpu_pct"]
    - df_metrics_sorted.groupby("server_id")["cpu_pct"].shift(6)
)

# Remove NaN rows for clean stats
clean_df = df_metrics_sorted.dropna(subset=["cpu_change_1h"])

# Volatility metrics per server
volatility = clean_df.groupby("server_id").agg(
    mean_abs_change=("cpu_change_1h", lambda x: x.abs().mean()),
    std_change=("cpu_change_1h", "std"),
    max_spike=("cpu_change_1h", lambda x: x.abs().max())
)

# Composite volatility score (better AIOps metric)
volatility["volatility_score"] = (
    0.5 * volatility["mean_abs_change"] +
    0.3 * volatility["std_change"] +
    0.2 * volatility["max_spike"]
)

# Rank servers
volatility = volatility.sort_values("volatility_score", ascending=False)

print(volatility)

print("\nMost volatile server:")
print(volatility.index[0])

           mean_abs_change  std_change  max_spike  volatility_score
server_id                                                          
srv-01           29.227273   35.840518       72.7         39.905792
srv-02           28.581818   34.830317       73.2         39.380004
srv-03           29.079798   34.760275       71.3         39.227982
srv-04           26.641414   32.722630       73.6         37.857496
srv-05           25.452525   31.385696       71.9         36.521971

Most volatile server:
srv-01


In [20]:
# Task 3 — server_metrics
# Create all 4 binary flags — cpu_critical, memory_critical, disk_warning, response_slow. Add stress_count. How many readings have stress_count >= 3? Which server has the most?

In [21]:
# SORT DATA

df_metrics_sorted = df_metrics.sort_values(
    ["server_id", "timestamp"]
)

# CREATE BINARY FLAGS

df_metrics_sorted["cpu_critical"] = (
    df_metrics_sorted["cpu_pct"] > 85
).astype(int)

df_metrics_sorted["memory_critical"] = (
    df_metrics_sorted["memory_pct"] > 85
).astype(int)

df_metrics_sorted["disk_warning"] = (
    df_metrics_sorted["disk_pct"] > 80
).astype(int)

df_metrics_sorted["response_slow"] = (
    df_metrics_sorted["response_ms"] > 900
).astype(int)

# -----------------------------------
# STRESS COUNT
# -----------------------------------

df_metrics_sorted["stress_count"] = (
    df_metrics_sorted[
        [
            "cpu_critical",
            "memory_critical",
            "disk_warning",
            "response_slow"
        ]
    ]
    .sum(axis=1)
)
# READINGS WITH STRESS >= 3

high_stress = df_metrics_sorted[
    df_metrics_sorted["stress_count"] >= 3
]

print("Total readings with stress_count >= 3:")
print(len(high_stress))

# WHICH SERVER HAS MOST?

server_stress_counts = (
    high_stress
    .groupby("server_id")
    .size()
    .sort_values(ascending=False)
)

print("\nStress readings per server:")
print(server_stress_counts)

print("\nServer with most high-stress readings:")
print(server_stress_counts.idxmax())

Total readings with stress_count >= 3:
20

Stress readings per server:
server_id
srv-05    7
srv-01    6
srv-03    3
srv-04    3
srv-02    1
dtype: int64

Server with most high-stress readings:
srv-05


In [24]:
# Task 4 — server_metrics
# Add time features — hour, day_of_week, is_business_hours. Compare mean cpu_pct during business hours vs off hours per server. Is CPU higher during business hours?

In [26]:
# Ensure timestamp is datetime
df_metrics["timestamp"] = pd.to_datetime(df_metrics["timestamp"])
df_metrics["hour"] = (df_metrics["timestamp"].dt.hour)

df_metrics["day_of_week"] = (df_metrics["timestamp"].dt.day_name())
df_metrics["is_business_hours"] = ((df_metrics["timestamp"].dt.weekday < 5) &  (df_metrics["hour"] >= 9)   &  (df_metrics["hour"] < 18))
cpu_comparison = ( df_metrics.groupby(["server_id", "is_business_hours"])["cpu_pct"].mean().unstack())

# Rename columns
cpu_comparison.columns = ["off_hours_cpu","business_hours_cpu"]
cpu_comparison["difference"] = ( cpu_comparison["business_hours_cpu"] - cpu_comparison["off_hours_cpu"])
cpu_comparison["business_hours_higher"] = (cpu_comparison["difference"] > 0)

# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

print(cpu_comparison)

# Servers where business-hour CPU is higher
higher_cpu_servers = cpu_comparison[cpu_comparison["business_hours_higher"]]

print("\nServers with higher CPU during business hours:")
print(higher_cpu_servers.index.tolist())

           off_hours_cpu  business_hours_cpu  difference  \
server_id                                                  
srv-01         62.558621           47.205556  -15.353065   
srv-02         60.508046           57.144444   -3.363602   
srv-03         59.465517           54.216667   -5.248851   
srv-04         58.996552           61.405556    2.409004   
srv-05         61.886207           64.488889    2.602682   

           business_hours_higher  
server_id                         
srv-01                     False  
srv-02                     False  
srv-03                     False  
srv-04                      True  
srv-05                      True  

Servers with higher CPU during business hours:
['srv-04', 'srv-05']
